In [1]:
import os
import sys
import json

try:
    import google.colab
    from google.colab import drive
    from google.colab import userdata
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("We are in Colab!")
    drive.mount('/content/drive', force_remount=True)
    
    !pip install -q tqdm requests openpyxl pandas scikit-learn
    
    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/restore_punct"
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
        
    try:
        API_KEY = userdata.get('YANDEX_API_KEY')
        FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')
    except userdata.SecretNotFoundError:
        raise ValueError("Please set YANDEX_API_KEY and YANDEX_FOLDER_ID in Colab Secrets!")

else:
    print("We are running locally!")
    ROOT_DIR = os.getcwd()
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
        
    secrets_path = os.path.join(ROOT_DIR, "yandex_secrets.json")
    if os.path.exists(secrets_path):
        with open(secrets_path, "r", encoding="utf-8") as f:
            secrets = json.load(f)
            API_KEY = secrets.get("YANDEX_API_KEY")
            FOLDER_ID = secrets.get("YANDEX_FOLDER_ID")
            if not API_KEY or not FOLDER_ID:
                raise ValueError("YANDEX_API_KEY or YANDEX_FOLDER_ID is missing in yandex_secrets.json")
    else:
        raise FileNotFoundError(f"Please create {secrets_path} with your API keys!")

DATA_DIR = os.path.join(ROOT_DIR, "data")

os.makedirs(DATA_DIR, exist_ok=True)

We are running locally!


In [2]:
import time
import requests
from tqdm.auto import tqdm

try:
    from extract_labels_standardized import extract_labels_standardized
except ImportError:
    raise ImportError("Error: Could not import extract_labels_standardized. Make sure the file exists in your ROOT_DIR.")

OUTPUT_FILE = os.path.join(DATA_DIR, "synthetic_rare_punct.json")
ITERATIONS_PER_PROMPT = 20

In [3]:
# PROMPTS =[
#     {
#         "domain": "fiction_dialogues",
#         "prompt": (
#             "Напиши 5 абсолютно независимых друг от друга отрывков из художественных произведений (каждый по 100 слов). "
#             "Сюжеты и герои во всех 5 отрывках должны полностью отличаться. "
#             "В каждом отрывке должна быть напряженная сцена диалога. Обязательно используй прямую речь, эмоциональные вопросы и восклицания. "
#             "Используй конструкции, где слова автора разрывают прямую речь. "
#             "Строго разделяй отрывки между собой тремя дефисами (---). Не используй нумерацию."
#         )
#     },
#     {
#         "domain": "science_legal",
#         "prompt": (
#             "Напиши 5 абсолютно независимых друг от друга отрывков из строгих научных статей или юридических документов (каждый по 100 слов). "
#             "В каждом отрывке обязательно используй длинные перечисления, разделенные точкой с запятой. "
#             "Также используй цитирование терминов в кавычках, включая вложенные кавычки (одни кавычки внутри других). "
#             "Строго разделяй отрывки между собой тремя дефисами (---). Не используй нумерацию."
#         )
#     },
#     {
#         "domain": "journalism_interview",
#         "prompt": (
#             "Напиши 5 абсолютно независимых друг от друга отрывков из газетных репортажей или интервью (каждый по 100 слов). "
#             "Обязательно используй риторические вопросы, тире перед началом реплик и внутри предложений, "
#             "а также цитирование собеседника в кавычках. "
#             "Строго разделяй отрывки между собой тремя дефисами (---). Не используй нумерацию."
#         )
#     }
# ]


In [4]:
PROMPTS =[
    {
        "domain": "fairy_tale_dialogue",
        "prompt": (
            "Ты — известный детский писатель. Напиши одну цельную, увлекательную сценку (150-200 слов), "
            "где двое сказочных героев эмоционально о чем-то спорят. Текст должен быть слитным рассказом, "
            "разделенным на абзацы. "
            "Ты ОБЯЗАН органично вплести в этот текст следующие конструкции: "
            "1) Прямую речь, которая заканчивается вопросом (Схема: «...?» — спросил он). "
            "2) Прямую речь, которая заканчивается восклицанием (Схема: «...!» — крикнул он). "
            "3) Слова автора, разрывающие фразу героя (Схема: «..., — сказал он, — ...»). "
            "Пиши простым, живым языком. Никаких списков, пунктов или нумерации."
        )
    },
    {
        "domain": "science_nested_quotes",
        "prompt": (
            "Ты — автор школьной познавательной энциклопедии. Напиши увлекательный, слитный абзац (150-200 слов) "
            "про изобретение роботов или космос. "
            "Ты ОБЯЗАН использовать в тексте: "
            "1) Длинное перечисление фактов, разделенных точкой с запятой (;). "
            "2) Цитату ученого, внутри которой есть еще одни кавычки (Пример: Ученый объяснил: «Планета \"Марс\" очень холодная»). "
            "Текст должен читаться как единый рассказ. Не используй списки."
        )
    }
]

In [5]:
def generate_yandex_text(prompt_text, temperature=0.7):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}",
        "x-folder-id": FOLDER_ID
    }
    
    body = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": "2000"
        },
        "messages":[
            {
                "role": "system", 
                "text": "Ты профессиональный редактор. Строго соблюдай инструкции по форматированию (разделитель ---) и пунктуации."
            },
            {"role": "user", "text": prompt_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=body)
        if response.status_code == 200:
            return response.json()['result']['alternatives'][0]['message']['text']
        else:
            print(f"API Error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"Request failed: {e}")
        return None


In [6]:
synthetic_dataset =[]

for prompt_info in PROMPTS:
    domain = prompt_info["domain"]
    p_text = prompt_info["prompt"]
    
    print(f"\n--- Generating domain: {domain} ---")
    for i in tqdm(range(ITERATIONS_PER_PROMPT), desc=f"Calls for {domain}"):
        generated_text = generate_yandex_text(p_text)
        
        if not generated_text:
            time.sleep(2)
            continue
            
        # Разрезаем ответ по разделителю
        snippets =[s.strip() for s in generated_text.split("---") if len(s.strip()) > 50]
        
        for snippet in snippets:
            try:
                pairs = extract_labels_standardized(snippet)
                
                if len(pairs) > 15: 
                    tokens = [p['word'] for p in pairs]
                    ner_tags = [p['label'] for p in pairs]
                    
                    synthetic_dataset.append({
                        "tokens": tokens,
                        "ner_tags": ner_tags,
                        "domain": domain
                    })
            except Exception as e:
                print(f"Parsing error: {e}")
                
        time.sleep(1.0)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(synthetic_dataset, f, ensure_ascii=False, indent=2)
    
print(f"\nSuccess! Extracted {len(synthetic_dataset)} valid synthetic snippets.")
print(f"Saved to {OUTPUT_FILE}.")



--- Generating domain: fairy_tale_dialogue ---


Calls for fairy_tale_dialogue:   0%|          | 0/20 [00:00<?, ?it/s]


--- Generating domain: science_nested_quotes ---


Calls for science_nested_quotes:   0%|          | 0/20 [00:00<?, ?it/s]


Success! Extracted 40 valid synthetic snippets.
Saved to /home/temari/god please no diploma/restore_punct/data/synthetic_rare_punct.json.


In [9]:
# Demonstrate synthesized examples from each domain
from collections import defaultdict
import pandas as pd
from extract_labels_standardized import ID_TO_LABEL

# Group examples by domain
examples_by_domain = defaultdict(list)
for item in synthetic_dataset:
    examples_by_domain[item["domain"]].append(item)

# Reconstruct text with punctuation for display
def reconstruct_text_with_punct(tokens, ner_tags):
    """Reconstruct text with punctuation from tokens and labels."""
    text = ""
    for token, tag in zip(tokens, ner_tags):
        # Get punctuation label (skip 0 which is "O" - no punctuation)
        label_name = ID_TO_LABEL.get(tag, "O")
        
        # Add space before token (except for first token)
        if text and not text.endswith(" "):
            text += " "
        
        text += token
        
        # Add punctuation after token if label is not "O"
        if label_name != "O" and label_name != "":
            text += label_name
    
    return text

# Display 2 examples from each domain
print("=" * 80)
print("SYNTHETIC DATA EXAMPLES (2 per domain)")
print("=" * 80)

for domain_name in ["fairy_tale_dialogue", "science_nested_quotes"]:
    examples = examples_by_domain[domain_name][:2]  # Get first 2 examples
    print(f"\n### Domain: {domain_name.upper()} ###\n")
    
    for idx, example in enumerate(examples, 1):
        tokens = example["tokens"]
        ner_tags = example["ner_tags"]
        
        # Reconstruct text with punctuation
        text_with_punct = reconstruct_text_with_punct(tokens, ner_tags)
        
        print(f"Example {idx}:")
        print(f"Text: {text_with_punct}\n")
        
        # Show as table
        df = pd.DataFrame({
            "Token": tokens[:30] if len(tokens) > 30 else tokens,  # Limit to first 30 for readability
            "Punctuation": [ID_TO_LABEL.get(tag, "O") for tag in ner_tags[:30]]
        })
        print(df.to_string(index=False))
        print()
        print("-" * 80)

print(f"\nTotal examples generated per domain:")
for domain_name in ["fairy_tale_dialogue", "science_nested_quotes"]:
    count = len(examples_by_domain[domain_name])
    print(f"  {domain_name}: {count}")


SYNTHETIC DATA EXAMPLES (2 per domain)

### Domain: FAIRY_TALE_DIALOGUE ###

Example 1:
Text: В густом лесу, где солнечные лучи пробивались сквозь густые ветви, встретились два необычных героя: весёлый гном по имени Звяк и мудрый эльф по имени Лир. Они сидели на большом пне и обсуждали, кто из них лучше знает лес. " Я знаю все пещеры и тоннели в этом лесу! - воскликнул Звяк. - Я могу найти золото и драгоценные камни там, где никто другой и не подумает искать!" Лир улыбнулся и ответил: " Это, конечно, замечательно, но ты же не можешь знать все тайны леса. Я, например, умею разговаривать с деревьями и понимать язык птиц Да что ты говоришь? - перебил его Звяк. - А ты знаешь, где спрятан древний клад, который ищут все гномы Конечно, знаю! - ответил Лир. - Но я не стану тебе его показывать. Ведь это не просто золото, это часть истории нашего леса". Звяк на мгновение задумался, а затем сказал: " Ты, конечно, мудр, Лир, но я всё равно считаю, что золото и драгоценные камни- это самое важное в

In [8]:
# Merge synthetic data with train_all.json
import random

train_all_path = os.path.join(DATA_DIR, "train_all.json")
train_all_synthetic_path = os.path.join(DATA_DIR, "train_all_synthetic.json")

random_seed = 42

# Load existing train_all.json
with open(train_all_path, 'r', encoding='utf-8') as f:
    train_all_data = json.load(f)

print(f"Loaded train_all.json: {len(train_all_data)} examples")
print(f"Loaded synthetic data: {len(synthetic_dataset)} examples")

# Merge: keep only 'tokens' and 'ner_tags' fields for consistency
merged_dataset = []

# Add original training data
for item in train_all_data:
    merged_dataset.append({
        "tokens": item["tokens"],
        "ner_tags": item["ner_tags"]
    })

# Add synthetic data (remove 'domain' field for consistency)
for item in synthetic_dataset:
    merged_dataset.append({
        "tokens": item["tokens"],
        "ner_tags": item["ner_tags"]
    })

# Shuffle with random seed for reproducibility
random.seed(random_seed)
random.shuffle(merged_dataset)

# Save merged and shuffled dataset
with open(train_all_synthetic_path, 'w', encoding='utf-8') as f:
    json.dump(merged_dataset, f, ensure_ascii=False, indent=2)

print(f"\n✅ Successfully merged and shuffled (seed={random_seed})!")
print(f"Total examples in train_all_synthetic.json: {len(merged_dataset)}")
print(f"Saved to: {train_all_synthetic_path}")


Loaded train_all.json: 61243 examples
Loaded synthetic data: 40 examples

✅ Successfully merged and shuffled (seed=42)!
Total examples in train_all_synthetic.json: 61283
Saved to: /home/temari/god please no diploma/restore_punct/data/train_all_synthetic.json
